In [ ]:
!pip install datasets
!pip install transformers
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 18.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system 

In [ ]:
from datasets import load_dataset
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import torch
from jiwer import wer


In [ ]:
librispeech = load_dataset("RaphaelOlivier/librispeech_asr_adversarial", "adv", split='natural')

model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h").to("cuda")
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
import IPython.display as ipd
example = librispeech[10]

audio_array = example['audio']['array']

display(ipd.Audio(audio_array, rate=16000))
print(example["true_text"])



IT IS THIS THAT IS OF INTEREST TO THEORY OF KNOWLEDGE


In [ ]:
from utils import transcribe_audio

In [ ]:
predicted_transcription = transcribe_audio(audio_array, 16000, processor, model)
print(predicted_transcription)

IT IS THIS THAT IS OF INTEREST TO THEORY OF KNOWLEDGE


In [ ]:
from fgsm import fgsm_attack

In [168]:
example = librispeech[11]
audio_array = example["audio"]["array"]  # Raw audio waveform
ground_truth = example["true_text"]  # Ground truth transcription
target_transcription = "HELLO WORLD"  # Target transcription

# Run FGSM attack
adversarial_waveform, ground_truth_wer, target_wer, adversarial_transcription = fgsm_attack(
    audio_array=audio_array,
    ground_truth=ground_truth,
    target_transcription=target_transcription,
    model=model,
    processor=processor,
    epsilon=0.02
)

# Print results
print(f"Ground Truth             : {ground_truth}")
print(f"Adversarial Transcription: {adversarial_transcription}")
print(f"WER (Ground Truth): {ground_truth_wer:.2f}")

Ground Truth             : AS USED IN THE SPEECH OF EVERYDAY LIFE THE WORD CARRIES AN UNDERTONE OF DEPRECATION
Adversarial Transcription: AS USE IN THE SPEECH OF EVERYDAY LIFE THE WORD CARIES AN UNDERTONE OF DEPRECATION
WER (Ground Truth): 0.13


In [169]:
import IPython.display as ipd
print("Original Audio:")
display(ipd.Audio(audio_array, rate=16000))
adversarial_audio = adversarial_waveform
print("Adversarial Audio:")
display(ipd.Audio(adversarial_audio, rate=16000))

Original Audio:


Adversarial Audio:


In [172]:
from statistics import mean
import json



# Define epsilon values to test (from 0.001 to 0.3)
epsilon_values = [0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3]

# Define target transcription
target_transcription = "HELLO WORLD"

# Store results
results = []
wer_by_epsilon = {eps: {"ground_truth_wer": [], "target_wer": []} for eps in epsilon_values}

# Select three samples for audio playback (e.g., indices 0, 1, 2)
audio_samples_to_play = [0, 1, 2]  # Adjust if you want different indices
audio_results = {idx: {eps: {} for eps in epsilon_values} for idx in audio_samples_to_play}

# Loop over the dataset
for idx, example in enumerate(librispeech):
    audio_array = example["audio"]["array"]  # Raw audio waveform
    ground_truth = example["true_text"]  # Ground truth transcription

    if idx % 10 == 0:
      print(f"\nSample {idx}")

    # Loop over epsilon values
    for epsilon in epsilon_values:
        # Run FGSM attack
        adversarial_waveform, ground_truth_wer, target_wer, adversarial_transcription = fgsm_attack(
            audio_array=audio_array,
            ground_truth=ground_truth,
            target_transcription=target_transcription,
            model=model,
            processor=processor,
            epsilon=epsilon
        )

        # Store result (exclude waveform to avoid JSON serialization issue)
        result = {
            "sample_idx": idx,
            "epsilon": epsilon,
            "ground_truth": ground_truth,
            "adversarial_transcription": adversarial_transcription,
            "ground_truth_wer": ground_truth_wer,
            "target_wer": target_wer
        }
        results.append(result)

        # Collect WERs for averaging
        wer_by_epsilon[epsilon]["ground_truth_wer"].append(ground_truth_wer)
        wer_by_epsilon[epsilon]["target_wer"].append(target_wer)

        # Store audio results for playback if sample is selected
        if idx in audio_samples_to_play:
            audio_results[idx][epsilon] = {
                "adversarial_waveform": adversarial_waveform,
                "ground_truth": ground_truth,
                "adversarial_transcription": adversarial_transcription,
                "ground_truth_wer": ground_truth_wer,
                "target_wer": target_wer
            }

# Play adversarial waveforms for selected samples
print("\nPlaying Adversarial Waveforms for Selected Samples:")
for idx in audio_samples_to_play:
    print(f"\nSample {idx}:")
    for epsilon in epsilon_values:
        audio_data = audio_results[idx][epsilon]
        print(f"\nEpsilon: {epsilon}")
        print(f"Ground Truth: {audio_data['ground_truth']}")
        print(f"Adversarial Transcription: {audio_data['adversarial_transcription']}")
        print(f"WER (Ground Truth): {audio_data['ground_truth_wer']:.2f}")
        print(f"WER (Target): {audio_data['target_wer']:.2f}")
        print("Playing Adversarial Audio:")
        display(ipd.Audio(audio_data['adversarial_waveform'], rate=16000))

# Compute and print average WER for each epsilon
print("\nAverage WER Across All Samples:")
for epsilon in epsilon_values:
    avg_ground_truth_wer = mean(wer_by_epsilon[epsilon]["ground_truth_wer"])
    avg_target_wer = mean(wer_by_epsilon[epsilon]["target_wer"])
    print(f"Epsilon: {epsilon}")
    print(f"  Average Ground Truth WER: {avg_ground_truth_wer:.2f}")
    print(f"  Average Target WER: {avg_target_wer:.2f}")

# Save results to JSON
with open("fgsm_results.json", "w") as f:
    json.dump(results, f, indent=2)


Sample 0

Sample 10

Sample 20

Sample 30

Sample 40

Sample 50

Sample 60

Sample 70

Sample 80

Playing Adversarial Waveforms for Selected Samples:

Sample 0:

Epsilon: 0.001
Ground Truth: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
Adversarial Transcription: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
WER (Ground Truth): 0.00
WER (Target): 4.00
Playing Adversarial Audio:



Epsilon: 0.005
Ground Truth: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
Adversarial Transcription: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
WER (Ground Truth): 0.00
WER (Target): 4.00
Playing Adversarial Audio:



Epsilon: 0.01
Ground Truth: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
Adversarial Transcription: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
WER (Ground Truth): 0.00
WER (Target): 4.00
Playing Adversarial Audio:



Epsilon: 0.02
Ground Truth: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
Adversarial Transcription: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
WER (Ground Truth): 0.00
WER (Target): 4.00
Playing Adversarial Audio:



Epsilon: 0.05
Ground Truth: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
Adversarial Transcription: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
WER (Ground Truth): 0.00
WER (Target): 4.00
Playing Adversarial Audio:


...


Epsilon: 0.05
Ground Truth: BUT I DO NOT THINK SUCH AN INFERENCE IS WARRANTED
Adversarial Transcription: BUT I DO NOT THINK SUCH AN IPERENCE IS WARRANTED
WER (Ground Truth): 0.10
WER (Target): 5.00
Playing Adversarial Audio:



Epsilon: 0.1
Ground Truth: BUT I DO NOT THINK SUCH AN INFERENCE IS WARRANTED
Adversarial Transcription: BUT I DO NOT THINK SUCH AN IPERENCE IS WARRANTED
WER (Ground Truth): 0.10
WER (Target): 5.00
Playing Adversarial Audio:



Epsilon: 0.2
Ground Truth: BUT I DO NOT THINK SUCH AN INFERENCE IS WARRANTED
Adversarial Transcription: BUT I DO NOT THINK SUCH AN EQENT IS WANTED
WER (Ground Truth): 0.20
WER (Target): 5.00
Playing Adversarial Audio:



Epsilon: 0.3
Ground Truth: BUT I DO NOT THINK SUCH AN INFERENCE IS WARRANTED
Adversarial Transcription: BUT I DO NOT THINK SUCH AMAIN IS WANDFUL
WER (Ground Truth): 0.30
WER (Target): 4.50
Playing Adversarial Audio:



Average WER Across All Samples:
Epsilon: 0.001
  Average Ground Truth WER: 0.06
  Average Target WER: 6.50
Epsilon: 0.005
  Average Ground Truth WER: 0.08
  Average Target WER: 6.49
Epsilon: 0.01
  Average Ground Truth WER: 0.09
  Average Target WER: 6.48
Epsilon: 0.02
  Average Ground Truth WER: 0.10
  Average Target WER: 6.47
Epsilon: 0.05
  Average Ground Truth WER: 0.10
  Average Target WER: 6.48
Epsilon: 0.1
  Average Ground Truth WER: 0.11
  Average Target WER: 6.49
Epsilon: 0.2
  Average Ground Truth WER: 0.17
  Average Target WER: 6.49
Epsilon: 0.3
  Average Ground Truth WER: 0.29
  Average Target WER: 6.44
